In [7]:
# 01_test_yandex_gpt.ipynb

import sys
import os
from pathlib import Path

# Добавляем корень проекта в path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Загружаем переменные окружения
from dotenv import load_dotenv
load_dotenv(project_root / '.env')

print("📁 Текущая директория:", os.getcwd())
print("📁 Корень проекта:", project_root)

📁 Текущая директория: /home/ruslan/RAG_PTE/notebooks
📁 Корень проекта: /home/ruslan/RAG_PTE


In [2]:
# Проверяем, что ключи загрузились
folder_id = os.getenv("YANDEX_FOLDER_ID")
api_key = os.getenv("YANDEX_API_KEY")

print(f"YANDEX_FOLDER_ID: {folder_id[:10]}...**" if folder_id else "❌ не найден")
print(f"YANDEX_API_KEY: {api_key[:10]}...**" if api_key else "❌ не найден")

YANDEX_FOLDER_ID: b1gq1o1t1a...**
YANDEX_API_KEY: AQVN3yNhEl...**


In [3]:
# Прямой запрос к Yandex GPT (без дополнительных библиотек)
import requests
import json

def test_yandex_gpt_simple():
    folder_id = os.getenv("YANDEX_FOLDER_ID")
    api_key = os.getenv("YANDEX_API_KEY")
    
    if not folder_id or not api_key:
        return "❌ Ключи не найдены"
    
    url = "https://llm.api.cloud.yandex.net/foundationModels/v1/completion"
    
    headers = {
        "Authorization": f"Api-Key {api_key}",
        "Content-Type": "application/json"
    }
    
    data = {
        "modelUri": f"gpt://{folder_id}/yandexgpt/latest",
        "completionOptions": {
            "stream": False,
            "temperature": 0.1,
            "maxTokens": 100
        },
        "messages": [
            {"role": "user", "text": "Привет! Ответь 'OK' если ты работаешь"}
        ]
    }
    
    try:
        response = requests.post(url, headers=headers, json=data, timeout=30)
        
        if response.status_code == 200:
            result = response.json()
            answer = result['result']['alternatives'][0]['message']['text']
            return f"✅ Успех! Ответ: {answer}"
        else:
            return f"❌ Ошибка {response.status_code}: {response.text}"
    except Exception as e:
        return f"❌ Исключение: {e}"

result = test_yandex_gpt_simple()
print(result)

✅ Успех! Ответ: OK, я работаю.


In [4]:
# Проверка с моделью 20B параметров (yandexgpt)
def test_yandex_gpt_20b():
    folder_id = os.getenv("YANDEX_FOLDER_ID")
    api_key = os.getenv("YANDEX_API_KEY")
    
    url = "https://llm.api.cloud.yandex.net/foundationModels/v1/completion"
    
    headers = {
        "Authorization": f"Api-Key {api_key}",
        "Content-Type": "application/json"
    }
    
    # Используем yandexgpt (20B параметров)
    data = {
        "modelUri": f"gpt://{folder_id}/yandexgpt/latest",
        "completionOptions": {
            "stream": False,
            "temperature": 0.1,
            "maxTokens": 200
        },
        "messages": [
            {"role": "system", "text": "Ты эксперт по правилам технической эксплуатации железных дорог (ПТЭ). Отвечай кратко и по существу."},
            {"role": "user", "text": "Какой пункт ПТЭ регулирует замыкание стрелочных переводов при запрещающих сигналах светофоров?"}
        ]
    }
    
    response = requests.post(url, headers=headers, json=data, timeout=30)
    
    if response.status_code == 200:
        result = response.json()
        answer = result['result']['alternatives'][0]['message']['text']
        print("✅ Yandex GPT 20B ответил:")
        print(f"Ответ: {answer}")
        return answer
    else:
        print(f"❌ Ошибка: {response.status_code}")
        print(response.text)
        return None

test_yandex_gpt_20b()

✅ Yandex GPT 20B ответил:
Ответ: Пункт 67 ПТЭ.


'Пункт 67 ПТЭ.'

In [5]:
# Проверка на вашем замечании про стрелочный перевод
def test_on_railway_remark():
    folder_id = os.getenv("YANDEX_FOLDER_ID")
    api_key = os.getenv("YANDEX_API_KEY")
    
    remark = """на железнодорожной станции систематически не замыкается по прямому направлению 
    стрелочный перевод № 47 при организации движения пассажирских поездов 
    по запрещающим показаниям светофоров."""
    
    url = "https://llm.api.cloud.yandex.net/foundationModels/v1/completion"
    
    headers = {
        "Authorization": f"Api-Key {api_key}",
        "Content-Type": "application/json"
    }
    
    prompt = f"""
Определи, какой пункт ПТЭ/ИДП нарушен в этой ситуации.

Ситуация: {remark}

Ответь строго в формате: "Приложение 14 пункт 15 ИДП"
Если считаешь что другой пункт - напиши его.
"""
    
    data = {
        "modelUri": f"gpt://{folder_id}/yandexgpt/latest",
        "completionOptions": {
            "stream": False,
            "temperature": 0.1,
            "maxTokens": 150
        },
        "messages": [
            {"role": "user", "text": prompt}
        ]
    }
    
    response = requests.post(url, headers=headers, json=data, timeout=30)
    
    if response.status_code == 200:
        result = response.json()
        answer = result['result']['alternatives'][0]['message']['text']
        print("📌 Замечание:", remark[:80] + "...")
        print("📌 Ответ Yandex GPT:")
        print(answer)
        return answer
    else:
        print(f"❌ Ошибка: {response.status_code}")
        print(response.text)
        return None

test_on_railway_remark()

📌 Замечание: на железнодорожной станции систематически не замыкается по прямому направлению 
...
📌 Ответ Yandex GPT:
Приложение 6 пункт 18 ИДП


'Приложение 6 пункт 18 ИДП'

In [6]:
# Улучшенная проверка с контекстом из ПТЭ
def test_with_context():
    folder_id = os.getenv("YANDEX_FOLDER_ID")
    api_key = os.getenv("YANDEX_API_KEY")
    
    # Контекст из реального ПТЭ (пункт 15)
    context = """
Приложение 14 пункт 15 ИДП. Когда невозможно открыть маневровые светофоры 
(или при отсутствии маневровых маршрутов) стрелки замыкаются специальными 
кнопками "замыкание стрелок" или управляющими командами (при их наличии на пульте управления).
"""
    
    remark = """на железнодорожной станции систематически не замыкается по прямому направлению 
    стрелочный перевод № 47 при организации движения пассажирских поездов 
    по запрещающим показаниям светофоров."""
    
    url = "https://llm.api.cloud.yandex.net/foundationModels/v1/completion"
    
    headers = {
        "Authorization": f"Api-Key {api_key}",
        "Content-Type": "application/json"
    }
    
    prompt = f"""
Ты эксперт по ПТЭ. Найди пункт, нарушенный в ситуации.

Контекст из документа:
{context}

Ситуация: {remark}

Какой пункт нарушен? Ответь только номером пункта.
"""
    
    data = {
        "modelUri": f"gpt://{folder_id}/yandexgpt/latest",
        "completionOptions": {
            "stream": False,
            "temperature": 0.1,
            "maxTokens": 100
        },
        "messages": [
            {"role": "user", "text": prompt}
        ]
    }
    
    response = requests.post(url, headers=headers, json=data, timeout=30)
    
    if response.status_code == 200:
        result = response.json()
        answer = result['result']['alternatives'][0]['message']['text']
        print(f"📌 Ответ с контекстом: {answer}")
        return answer
    else:
        print(f"❌ Ошибка: {response.status_code}")
        return None

test_with_context()

📌 Ответ с контекстом: 15


'15'